# Data Overview & Analysis

**Complete Data Pipeline Visualization**

Raw Chunks → SFT Candidates → DPO Pairs

Statistics, samples, quality metrics, and distributions.

## Setup

In [ ]:
import os
import sys
import json
from pathlib import Path
from collections import Counter, defaultdict

# Initialize project path (portable - works on any computer)
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    # Fallback: go up one level if src not in current directory
    PROJECT_ROOT = PROJECT_ROOT.parent
    if not (PROJECT_ROOT / 'src').exists():
        raise FileNotFoundError('Could not find src/ directory. Make sure you run this notebook from the project root.')

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from sft import load_chunks
from sft.schema import SFTSample

print(f'✓ Project: {PROJECT_ROOT.name}')

In [ ]:
# Define paths
CHUNKS_PATH = PROJECT_ROOT / 'data' / 'processed' / 'chunks_asas_albalagha.jsonl'
SFT_CANDIDATES = PROJECT_ROOT / 'data' / 'generated' / 'sft' / 'candidates.jsonl'
SFT_ACCEPTED = PROJECT_ROOT / 'data' / 'generated' / 'sft' / 'accepted.jsonl'
DPO_CANDIDATES = PROJECT_ROOT / 'data' / 'generated' / 'dpo' / 'candidates.jsonl'

print('\n=== DATA FILES ===')
print(f'Raw Chunks:      {CHUNKS_PATH.exists()} ({CHUNKS_PATH})')
print(f'SFT Candidates:  {SFT_CANDIDATES.exists()} ({SFT_CANDIDATES})')
print(f'SFT Accepted:    {SFT_ACCEPTED.exists()} ({SFT_ACCEPTED})')
print(f'DPO Candidates:  {DPO_CANDIDATES.exists()} ({DPO_CANDIDATES})')

## 1. Raw Chunks Overview

In [ ]:
print('\n=== RAW CHUNKS ANALYSIS ===')

chunks = load_chunks(str(CHUNKS_PATH))
print(f'Total chunks: {len(chunks):,}')

# Analyze chunk structure
regions = Counter(c.get('region') for c in chunks)
formats = Counter(c.get('format_type') for c in chunks)
chunk_sizes = [len(c.get('chunk_text', '')) for c in chunks]

print(f'\nRegions: {dict(regions)}')
print(f'Format types: {dict(formats)}')
print(f'\nChunk text length:')
print(f'  Min: {min(chunk_sizes):,} chars')
print(f'  Max: {max(chunk_sizes):,} chars')
print(f'  Avg: {sum(chunk_sizes)//len(chunk_sizes):,} chars')
print(f'  Total: {sum(chunk_sizes):,} chars')

In [ ]:
# Show sample chunks
print('\n=== SAMPLE CHUNKS ===')
for i in range(min(3, len(chunks))):
    chunk = chunks[i]
    print(f'\nChunk {i+1}: {chunk.get("chunk_id")}')
    print(f'  Region: {chunk.get("region")}')
    print(f'  Format: {chunk.get("format_type")}')
    print(f'  Size: {len(chunk.get("chunk_text", ""))} chars')
    print(f'  Text: {chunk.get("chunk_text", "")[:150]}...')

## 2. SFT Candidates Overview

In [ ]:
print('\n=== SFT CANDIDATES ANALYSIS ===')

sft_candidates = []
if SFT_CANDIDATES.exists():
    with open(SFT_CANDIDATES, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data = json.loads(line)
                sft_candidates.append(data)
    
    print(f'Total SFT candidates: {len(sft_candidates):,}')
    
    # Analyze distributions
    q_types = Counter(s.get('question_type') for s in sft_candidates)
    difficulties = Counter(s.get('difficulty') for s in sft_candidates)
    reasoning_modes = Counter(s.get('reasoning_mode') for s in sft_candidates)
    
    print(f'\nQuestion Types:')
    for qt, count in q_types.most_common():
        print(f'  {qt}: {count:,} ({count/len(sft_candidates)*100:.1f}%)')
    
    print(f'\nDifficulty Levels:')
    for diff, count in difficulties.most_common():
        print(f'  {diff}: {count:,} ({count/len(sft_candidates)*100:.1f}%)')
    
    print(f'\nReasoning Modes (top 5):')
    for mode, count in reasoning_modes.most_common(5):
        print(f'  {mode}: {count:,}')
else:
    print('⚠️  SFT candidates file not found')
    sft_candidates = []

In [ ]:
# Question and answer length analysis
if sft_candidates:
    q_lengths = [len(s.get('question', '')) for s in sft_candidates]
    a_lengths = [len(s.get('answer', '')) for s in sft_candidates]
    t_lengths = [len(s.get('thinking_arabic', '')) for s in sft_candidates]
    
    print('\n=== SFT TEXT LENGTHS ===')
    print(f'\nQuestion length (chars):')
    print(f'  Min: {min(q_lengths)}, Max: {max(q_lengths)}, Avg: {sum(q_lengths)//len(q_lengths)}')
    
    print(f'\nAnswer length (chars):')
    print(f'  Min: {min(a_lengths)}, Max: {max(a_lengths)}, Avg: {sum(a_lengths)//len(a_lengths)}')
    
    print(f'\nThinking length (chars):')
    print(f'  Min: {min(t_lengths)}, Max: {max(t_lengths)}, Avg: {sum(t_lengths)//len(t_lengths)}')
    
    empty_thinking = sum(1 for t in t_lengths if t == 0)
    print(f'  Empty thinking: {empty_thinking:,} ({empty_thinking/len(sft_candidates)*100:.1f}%)')

In [ ]:
# Show sample SFT candidates
if sft_candidates:
    print('\n=== SAMPLE SFT CANDIDATES ===')
    for i in range(min(2, len(sft_candidates))):
        s = sft_candidates[i]
        print(f'\nSample {i+1}: {s.get("sample_id")}')
        print(f'  Type: {s.get("question_type")} | Diff: {s.get("difficulty")}')
        print(f'  Q: {s.get("question")[:80]}...')
        print(f'  A: {s.get("answer")[:80]}...')
        print(f'  T(AR): {s.get("thinking_arabic")[:60]}...' if s.get("thinking_arabic") else '  T(AR): (empty)')
        print(f'  T(EN): {s.get("thinking_english")[:60]}...' if s.get("thinking_english") else '  T(EN): (empty)')

## 3. SFT Accepted Overview

In [ ]:
print('\n=== SFT ACCEPTED ANALYSIS ===')

sft_accepted = []
if SFT_ACCEPTED.exists():
    with open(SFT_ACCEPTED, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data = json.loads(line)
                sft_accepted.append(data)
    
    print(f'Total accepted: {len(sft_accepted):,}')
    if sft_candidates:
        accept_rate = len(sft_accepted) / len(sft_candidates) * 100
        print(f'Acceptance rate: {accept_rate:.1f}%')
    
    # Same analysis as candidates
    q_types_acc = Counter(s.get('question_type') for s in sft_accepted)
    difficulties_acc = Counter(s.get('difficulty') for s in sft_accepted)
    
    print(f'\nQuestion Types (Accepted):')
    for qt, count in q_types_acc.most_common():
        print(f'  {qt}: {count:,} ({count/len(sft_accepted)*100:.1f}%)')
    
    print(f'\nDifficulty Levels (Accepted):')
    for diff, count in difficulties_acc.most_common():
        print(f'  {diff}: {count:,} ({count/len(sft_accepted)*100:.1f}%)')
else:
    print('⚠️  SFT accepted file not found')
    sft_accepted = []

## 4. DPO Pairs Overview

In [ ]:
print('\n=== DPO CANDIDATES ANALYSIS ===')

dpo_candidates = []
if DPO_CANDIDATES.exists():
    with open(DPO_CANDIDATES, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data = json.loads(line)
                dpo_candidates.append(data)
    
    print(f'Total DPO pairs: {len(dpo_candidates):,}')
    if sft_accepted:
        dpo_rate = len(dpo_candidates) / len(sft_accepted) * 100
        print(f'DPO generation rate: {dpo_rate:.1f}%')
    
    # Analyze rejection types
    rejection_types = Counter(p.get('rejection_type') for p in dpo_candidates)
    
    print(f'\nRejection Types:')
    for rej_type, count in rejection_types.most_common():
        print(f'  {rej_type}: {count:,} ({count/len(dpo_candidates)*100:.1f}%)')
else:
    print('⚠️  DPO candidates file not found')
    dpo_candidates = []

In [ ]:
# Show sample DPO pairs
if dpo_candidates:
    print('\n=== SAMPLE DPO PAIRS ===')
    for i in range(min(2, len(dpo_candidates))):
        p = dpo_candidates[i]
        print(f'\nPair {i+1}: {p.get("pair_id")}')
        print(f'  Rejection type: {p.get("rejection_type")}')
        
        prompt = p.get('prompt', [{}])[0].get('content', '')
        chosen = p.get('chosen', [{}])[0].get('content', '')
        rejected = p.get('rejected', [{}])[0].get('content', '')
        
        print(f'  Prompt: {prompt[:60]}...')
        print(f'  Chosen: {chosen[:60]}...')
        print(f'  Rejected: {rejected[:60]}...')

## 5. Pipeline Summary

In [ ]:
print('\n' + '='*60)
print('DATA PIPELINE SUMMARY')
print('='*60)
print(f'Raw Chunks:       {len(chunks):>10,}')
print(f'SFT Candidates:   {len(sft_candidates):>10,}')
if sft_candidates:
    print(f'  → {len(sft_candidates)/len(chunks):.1f} samples per chunk')
print(f'SFT Accepted:     {len(sft_accepted):>10,}')
if sft_candidates:
    print(f'  → {len(sft_accepted)/len(sft_candidates)*100:.1f}% acceptance rate')
print(f'DPO Pairs:        {len(dpo_candidates):>10,}')
if sft_accepted:
    print(f'  → {len(dpo_candidates)/len(sft_accepted)*100:.1f}% conversion from SFT')
print('='*60)

## 6. Quality Insights

In [ ]:
print('\n=== KEY METRICS ===')

if sft_candidates and sft_accepted:
    print(f'\n✓ Generation Success:')
    print(f'  {len(sft_candidates):,} candidates from {len(chunks):,} chunks')
    print(f'  {len(sft_candidates)/len(chunks):.1f}x expansion rate')
    
    print(f'\n✓ Quality Filtering:')
    print(f'  {len(sft_accepted):,} passed confidence threshold')
    print(f'  {len(sft_candidates)-len(sft_accepted):,} filtered out')
    print(f'  {len(sft_accepted)/len(sft_candidates)*100:.1f}% pass rate')
    
    if dpo_candidates:
        print(f'\n✓ DPO Pair Generation:')
        print(f'  {len(dpo_candidates):,} preference pairs created')
        print(f'  {len(dpo_candidates)/len(sft_accepted)*100:.1f}% conversion rate')
        
        rejection_types = Counter(p.get('rejection_type') for p in dpo_candidates)
        print(f'  Top rejection type: {rejection_types.most_common(1)[0][0]}')

print('\n✓ All data loaded successfully!')